<a href="https://colab.research.google.com/github/iav2002/AppliedDeepLearning/blob/main/part2_DATAPREP_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applied Deep Learning, Part 2: Dataset Preparation
## 1. Setup

Produces the canonical splits used by every other notebook. Outputs five CSVs into `../data_splits/`.

Splits in plain English
1. test, 10% held out, never touched until final eval
2. train and val, 75% and 15% of the remainder, stratified


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
# rebuild the labels df, same logic as notebook 1
DATASET_ROOT = Path("face_age")

rows = []
for age_dir in sorted(DATASET_ROOT.iterdir()):
    if not age_dir.is_dir():
        continue
    age = int(age_dir.name)
    for f in age_dir.iterdir():
        if f.is_file():
            rows.append({"path": str(f), "age": age, "fname": f.name})

df = pd.DataFrame(rows)
print("loaded", len(df), "rows")

loaded 9778 rows


## 2. Categories and sparse age filter

Apply the 7 bins decided in EDA. Drop ages with fewer than 5 images (decided in EDA), this affects ages 91, 95, 99, 100, 101, 110 (those are the ones the EDA flagged).

In [ ]:
# bins from EDA, 7 categories
bins = [0, 5, 13, 20, 31, 46, 61, 200]
labels = ["infant", "child", "teen", "youth", "mid", "mature", "senior"]
df["age_category"] = pd.cut(df["age"], bins=bins, labels=labels, right=False)

# count images per age, then keep only ages with at least 5 samples
age_counts = df["age"].value_counts()
keep_ages = age_counts[age_counts >= 5].index
before = len(df)
df = df[df["age"].isin(keep_ages)].reset_index(drop=True)
after = len(df)
print(f"dropped {before - after} rows from sparse ages")
print(f"remaining {after} rows across {df['age'].nunique()} ages")
print("\ncategory counts after filter")
print(df["age_category"].value_counts().reindex(labels))

dropped 18 rows from sparse ages
remaining 9760 rows across 91 ages

category counts after filter
age_category
infant    2131
child     1124
teen       909
youth     1626
mid       1306
mature    1369
senior    1295
Name: count, dtype: int64
